# Google Colab Version: [Open this notebook in Google Colab](https://colab.research.google.com/github/parawaveio/parawave/blob/main/examples/02_synthetic_data_pipeline.ipynb)

# Synthetic Data Pipeline

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parawaveio/parawave/blob/main/examples/02_synthetic_data_pipeline.ipynb)

A common pattern: download a dataset from HuggingFace, process each item with an LLM in parallel.
Here we rate 100 movie reviews for sentiment and quality using parawave.

In [1]:
!pip install parawave datasets openai

  Using cached datasets-4.8.2-py3-none-any.whl.metadata (19 kB)


  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 2.9 MB/s eta 0:00:00


Using cached datasets-4.8.2-py3-none-any.whl (526 kB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 8.3 MB/s eta 0:00:00


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 0.0/34.3 MB ? eta -:--:--

...

   ━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.6/9.9 MB 75.0 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━ 9.1/9.9 MB 79.7 MB/s eta 0:00:01

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 73.3 MB/s eta 0:00:00


  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 20.0.0
    Uninstalling pyarrow-20.0.0:
      Successfully uninstalled pyarrow-20.0.0



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
# Set your OpenAI API key, or load from .env / environment variable
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "your-key-here")


## Load Data from HuggingFace

In [3]:
from datasets import load_dataset

ds = load_dataset("rotten_tomatoes", split="test[:100]")
texts = ds["text"]

print(f"Loaded {len(texts)} reviews")
print(f"Sample: {texts[0][:200]}...")

/Users/zhengisamazing/1.python_dir/parawaveio/parawave/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 100 reviews
Sample: lovingly photographed in the manner of a golden book sprung to life , stuart little 2 manages sweetness largely without stickiness ....


## Rate with Parawave

Call an LLM to rate each review for sentiment and quality, with concurrency, rate limiting, and automatic retry.

In [4]:
import parawave
from openai import AsyncOpenAI

client = AsyncOpenAI()


@parawave(
    max_concurrency=10,
    rate_limit=20,
    retry=parawave.RetryPolicy(max_retries=3, backoff="exponential", base_delay=0.5),
    progress="console",
)
async def rate_text(text: str) -> dict:
    response = await client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": "Rate the following movie review. Return JSON with: sentiment (positive/negative), quality (1-5), and a one-sentence summary."},
            {"role": "user", "content": text[:500]},  # truncate long reviews
        ],
        response_format={"type": "json_object"},
    )
    import json
    return json.loads(response.choices[0].message.content)


result = rate_text.run(data=[{"text": t} for t in texts])

[parawave] Starting run-252dc0e34142 | 100 items | concurrency=10


[parawave] 1/100 (1 completed) | 1.1s | 0.9 items/s


[parawave] 9/100 (9 completed) | 1.7s | 5.3 items/s


[parawave] 15/100 (15 completed) | 2.2s | 6.7 items/s


[parawave] 20/100 (20 completed) | 2.8s | 7.2 items/s


...

[parawave] 79/100 (79 completed) | 7.4s | 10.6 items/s


[parawave] 87/100 (87 completed) | 8.0s | 10.8 items/s


[parawave] 96/100 (96 completed) | 8.6s | 11.2 items/s


[parawave] 100/100 (100 completed) | 9.1s | 10.9 items/s


[parawave] Completed run-252dc0e34142 | 100/100 completed | 9.1s | 10.9 items/s


## Results

In [5]:
print(result.summary)
print()

for i in range(5):
    item = result[i]
    print(f"[{i}] {texts[i][:80]}...")
    print(f"    Rating: {item.output}")
    print()

100/100 completed | 9.1s | 10.9 items/s

[0] lovingly photographed in the manner of a golden book sprung to life , stuart lit...
    Rating: {'sentiment': 'positive', 'quality': 4, 'summary': 'The review praises the film as charmingly and lovingly photographed, emphasizing its sweetness without becoming overly sentimental.'}

[1] consistently clever and suspenseful ....
    Rating: {'sentiment': 'positive', 'quality': 4, 'summary': 'The review highlights that the movie is consistently clever and suspenseful, indicating a strong overall impression.'}

[2] it's like a " big chill " reunion of the baader-meinhof gang , only these guys a...
    Rating: {'sentiment': 'positive', 'quality': 3, 'summary': 'The review is generally favorable, describing the film as an amusing reunion-style story with harmless prankster energy rather than serious political activism.'}

[3] the story gives ample opportunity for large-scale action and suspense , which di...
    Rating: {'sentiment': 'positive', 'q

## Resume if needed

In [6]:
if not result.ok:
    print(f"{result.num_failed} items failed — resuming...")
    result = rate_text.resume()
    print(result.summary)

## Export

In [7]:
result.to_csv("rated_reviews.csv")
print(f"Saved {result.num_completed} rated reviews to rated_reviews.csv")

Saved 100 rated reviews to rated_reviews.csv
